### Tools Creation and Binding

In [1]:
from langchain_openai import ChatOpenAI
from langchain_community.tools import tool

In [2]:
@tool
def multipy_two_numbers(a: float, b: float) -> float:
    "This functions multiplies two numbers and return the result."
    return a * b

In [3]:
print(multipy_two_numbers.invoke({'a': 4, 'b': 5}))
print(multipy_two_numbers.description)
print(multipy_two_numbers.name)

20.0
This functions multiplies two numbers and return the result.
multipy_two_numbers


In [4]:
llm = ChatOpenAI()

llm_with_tool = llm.bind_tools([multipy_two_numbers])

### Tools Calling

- LLM does not actually run the tool, it suggests the tool and the input arguments. The actual execution is handled by LangChain or you. 

In [5]:
llm_with_tool.invoke('Can you multiply 19 with 5')

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 63, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DXALpK39VJpBWkGeFJvcyUdJORNcI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db15a-6394-7311-8801-aac4325c0540-0', tool_calls=[{'name': 'multipy_two_numbers', 'args': {'a': 19, 'b': 5}, 'id': 'call_mCKZiTAIXhpM1tSy9NVgFgt8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 21, 'total_tokens': 84, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
llm_with_tool.invoke('Can you multiply 19 with 5').tool_calls

[{'name': 'multipy_two_numbers',
  'args': {'a': 19, 'b': 5},
  'id': 'call_v1skqKvwseKgY77IUO5sF1xI',
  'type': 'tool_call'}]

### Tool Execution

In [7]:
multipy_two_numbers.invoke({'name': 'multipy_two_numbers',
  'args': {'a': 19, 'b': 5},
  'id': 'call_J0BWIpd7JPxWGylLb2pSKAwx',
  'type': 'tool_call'})

ToolMessage(content='95.0', name='multipy_two_numbers', tool_call_id='call_J0BWIpd7JPxWGylLb2pSKAwx')

In [10]:
from langchain_core.messages import HumanMessage

query = HumanMessage('Can you multiply 4 with 100')

message = [query]

print(message)
print("="*20)

result = llm_with_tool.invoke(message)

print(result)
print("="*20)

message.append(result)

tool_result = multipy_two_numbers.invoke(result.tool_calls[0])

print(tool_result)
print("="*20)

message.append(tool_result)

print(message)

print("="*20)
llm_with_tool.invoke(message)

[HumanMessage(content='Can you multiply 4 with 100', additional_kwargs={}, response_metadata={})]
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 63, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DXARkAPma3YGPPRD9fUM4C3M7mqsk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019db15f-fec5-7760-8aba-811a858f1a85-0' tool_calls=[{'name': 'multipy_two_numbers', 'args': {'a': 4, 'b': 100}, 'id': 'call_fT1X5xvrcwqniUPfP3qmnGj6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 63, 'output_tokens': 21, 'total_tokens': 84, 'input_token_details': {'audio': 0, 'cache_read'

AIMessage(content='The result of multiplying 4 with 100 is 400.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 97, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DXARkItvDsomCvrytjydqE8y8KPL6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019db160-0295-7691-829a-8056d8d23f29-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 14, 'total_tokens': 111, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Currency Conversion Tool

In [94]:
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [ ]:
@tool 
def get_conversion_rate(base_currency: str, target_currency: str) -> float:
    "This function return the conversion rate from one currency to another by calling an API."
    url = f"https://v6.exchangerate-api.com/v6/4330f5fe17d*****550df7276/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()

get_conversion_rate.invoke({'base_currency': 'USD', 'target_currency' : 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1776729601,
 'time_last_update_utc': 'Tue, 21 Apr 2026 00:00:01 +0000',
 'time_next_update_unix': 1776816001,
 'time_next_update_utc': 'Wed, 22 Apr 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 93.1663}

In [96]:
get_conversion_rate.invoke({'base_currency': 'USD', 'target_currency' : 'INR'}).get('conversion_rate')

93.1663

In [110]:
@tool
def convert_currency(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float: 
    "It converts the base_currency_value to target_currency using conversion_rate"
    return base_currency_value*conversion_rate

In [111]:
llm = ChatOpenAI()

In [117]:
llm_with_tools = llm.bind_tools([get_conversion_rate, convert_currency])

In [118]:
message = [HumanMessage("What is the conversion factor between USD and INR? And also convert 100 USD to INR using the get_conversion_rate tool?")]

In [119]:
message

[HumanMessage(content='What is the conversion factor between USD and INR? And also convert 100 USD to INR using the get_conversion_rate tool?', additional_kwargs={}, response_metadata={})]

In [120]:
ai_message = llm_with_tools.invoke(message)

print(ai_message.tool_calls)

message.append(ai_message)    

print('='*20)

ai_message

[{'name': 'get_conversion_rate', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_89NKH8Ck9C5B14terHaseNst', 'type': 'tool_call'}]


AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 118, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DXDYkN7Z7CkC0K17WcJwLdv35toge', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db216-9636-7772-bb6c-c69160cec2fd-0', tool_calls=[{'name': 'get_conversion_rate', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_89NKH8Ck9C5B14terHaseNst', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 118, 'output_tokens': 22, 'total_tokens': 140, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 're

In [90]:
ai_message.content

''

In [107]:
for tool_call in ai_message.tool_calls:
    print(tool_call)

{'name': 'get_conversion_rate', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_HfmQtZFZiUwy3XI84NLB5Pho', 'type': 'tool_call'}


In [106]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool call and get the conversion rate
    if tool_call['name'] == 'get_conversion_rate':
        tool_message1 = get_conversion_rate.invoke(tool_call)
        print(tool_message1)
        print('='*20)
        # fetch the conversion rate from the tool_message1
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        message.append(tool_message1) 
    # execute the 2nd tool call and get the converted currency
    if tool_call['name'] == 'convert_currency':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert_currency.invoke(tool_call['args'])
        print(tool_message2)
        print('='*20)
        message.append(tool_message2)

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1776729601, "time_last_update_utc": "Tue, 21 Apr 2026 00:00:01 +0000", "time_next_update_unix": 1776816001, "time_next_update_utc": "Wed, 22 Apr 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 93.1663}' name='get_conversion_rate' tool_call_id='call_HfmQtZFZiUwy3XI84NLB5Pho'


In [108]:
print(message)

[HumanMessage(content='What is the conversion factor between USD and INR? And also convert 100 USD to INR using that conversion factor?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 114, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DXDTXaJjFYb8nVBIYfJFlleBHfoht', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db211-a69e-7e90-8159-d5fa7db647dd-0', tool_calls=[{'name': 'get_conversion_rate', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_HfmQtZFZiUwy3XI84NLB5Pho', 'type': 'tool_call'}], invalid_tool_

In [121]:
# llm_with_tools.invoke(H_message)